In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

TICKERS=[
'NVDA', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NFLX', # Tech
    'LLY', 'JNJ', 'PFE', 'UNH', 'MRK',  # Healthcare
    'JPM', 'V', 'PYPL', 'GS',   # Finance
    'KO', 'WMT', 'PG', 'PEP',   # Consumer
    'BA', 'INTC', 'DIS', 'XOM'  # Diverse/Industrial/Energy
]

START_DATE='2022-01-01'
SPLIT_DATE='2025-01-01' # 3 years Train(2022-2024)
END_DATE='2026-01-01'    # 1 year Test(2025)
RISK_FREE_RATE=0.02 # Constant from Sharpe Ratio formula


# Define paths to local files
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, 'data')

TRAIN_FILE = os.path.join(DATA_DIR, 'train_returns.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test_returns.csv')
MU_TRAIN_FILE = os.path.join(DATA_DIR, 'mu_train.csv')
SIGMA_TRAIN_FILE = os.path.join(DATA_DIR, 'sigma_train.csv')
MU_TEST_FILE = os.path.join(DATA_DIR, 'mu_test.csv')
SIGMA_TEST_FILE = os.path.join(DATA_DIR, 'sigma_test.csv')



def get_processed_data():
    """
        Loads locally saved data (if available) or downloads from Yahoo Finance,
        processes it and saves it in the 'data' folder for later use.
    """

    # Create 'data' folder if it doesn't exist
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)

    # Check if all necessary files have already been saved
    if (os.path.exists(TRAIN_FILE) and os.path.exists(TEST_FILE) and
        os.path.exists(MU_TRAIN_FILE) and os.path.exists(SIGMA_TRAIN_FILE) and
        os.path.exists(MU_TEST_FILE) and os.path.exists(SIGMA_TEST_FILE)):
        print("Loading processed data from local 'data/' folder...")

        test_returns = pd.read_csv(TEST_FILE, index_col='Date', parse_dates=True)

        # Read mu and sigma and transform them back to numpy vectors/matrices (.values)
        mu_train = pd.read_csv(MU_TRAIN_FILE, index_col=0).squeeze("columns").values
        sigma_train = pd.read_csv(SIGMA_TRAIN_FILE, index_col=0).values
        mu_test = pd.read_csv(MU_TEST_FILE, index_col=0).squeeze("columns").values
        sigma_test = pd.read_csv(SIGMA_TEST_FILE, index_col=0).values

        return mu_train, sigma_train, mu_test, sigma_test

    print("Local data not found. Downloading from Yahoo Finance...")

    data = yf.download(TICKERS, start=START_DATE, end=END_DATE)['Close']

    # Calculate daily percentage returns
    returns = data.pct_change().dropna()

    # Split data: Train vs Test
    train_returns = returns.loc[:SPLIT_DATE]
    test_returns = returns.loc[SPLIT_DATE:]

    # Process for algorithms (In-Sample/Train) - multiply by 252 because there are 252 trading days per year
    mu_train_series = train_returns.mean() * 252
    sigma_train_df = train_returns.cov() * 252

    # Process for algorithms (Out-of-Sample/Test) - multiply by 252 for annualized values
    mu_test_series = test_returns.mean() * 252
    sigma_test_df = test_returns.cov() * 252

    # SAVE DATA TO 'data/' FOLDER
    print("Saving data to 'data/' folder...")
    train_returns.to_csv(TRAIN_FILE)
    test_returns.to_csv(TEST_FILE)
    mu_train_series.to_csv(MU_TRAIN_FILE, header=['Mu'])
    sigma_train_df.to_csv(SIGMA_TRAIN_FILE)
    mu_test_series.to_csv(MU_TEST_FILE, header=['Mu'])
    sigma_test_df.to_csv(SIGMA_TEST_FILE)

    print(f"Data processed and saved! Training: {len(train_returns)} days, Testing: {len(test_returns)} days.")
    return mu_train_series.values, sigma_train_df.values, mu_test_series.values, sigma_test_df.values, test_returns

def calculate_sharpe_ratio(weights, mu, sigma, risk_free_rate=RISK_FREE_RATE):
    """
        Objective Function (Fitness) that all 3 algorithms must maximize.
    """
    weights = np.array(weights)

    portfolio_return = np.sum(mu * weights)
    portfolio_variance = np.dot(weights.T, np.dot(sigma, weights))
    portfolio_risk = np.sqrt(portfolio_variance)

    if portfolio_risk == 0:
        return 0

    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_risk
    return sharpe_ratio

def validate_and_normalize_weights(weights):
    """
        Ensures the constraints are met: sum(weights) == 1 and weights >= 0.
    """
    weights = np.maximum(0, weights)
    total_weight = np.sum(weights)
    if total_weight > 0:
        return weights / total_weight
    else:
        return np.ones(len(weights)) / len(weights)


def calculate_markowitz_benchmark(mu, sigma, risk_free_rate=RISK_FREE_RATE):
    import scipy.optimize as scipy_optimize

    """
        Calculates the mathematically optimal portfolio using Modern Portfolio Theory (Harry Markowitz).
        This function applies the SLSQP (Sequential Least Squares Programming) optimization algorithm
        to find the exact asset weights that maximize the Sharpe Ratio.
        The result serves as a benchmark for evaluating heuristic algorithms such as Hybrid Genetic Algorithm (HGA), PSO, and SCA-BAS.

        Parameters:
            mu: vector of expected annual returns for each stock
            sigma: covariance matrix representing asset risk and correlations
            risk_free_rate: risk-free interest rate
        Returns:
            best_weights: optimal portfolio weights (percentages) for each stock
            best_sharpe_ratio: maximum theoretical Sharpe Ratio achievable with the given data
    """
    num_assets = len(mu)

    # Function for the mathematical optimizer to minimize
    # SciPy can only minimize functions, but we want to maximize Sharpe Ratio => we return -Sharpe Ratio
    # Maximizing X is equivalent to minimizing -X
    def negative_sharpe_ratio(weights):
        current_sharpe = calculate_sharpe_ratio(weights, mu, sigma, risk_free_rate)
        return -1 * current_sharpe

    # Define problem constraints
    constraints = ({
        'type': 'eq',
        'fun': lambda weights: np.sum(weights) - 1.0  # we want the sum of weights to be 1
    })

    # Define bounds for weights
    bounds = tuple((0.0, 1.0) for _ in range(num_assets))

    # Starting point: start from a perfectly distributed portfolio (4% for each of the 25 stocks)
    initial_weights = np.array(num_assets * [1.0 / num_assets])

    # Call mathematical optimization algorithm from SciPy
    optimization_result = scipy_optimize.minimize(
        fun=negative_sharpe_ratio,
        x0=initial_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    optimal_weights = optimization_result['x']

    # Calculate the real Sharpe Ratio for these weights
    max_sharpe_ratio = calculate_sharpe_ratio(optimal_weights, mu, sigma, risk_free_rate)

    return optimal_weights, max_sharpe_ratio

In [2]:
mu_train, sigma_train, mu_test, sigma_test = get_processed_data()

Loading processed data from local 'data/' folder...


In [3]:
import random
import time

class MemeticAlgorithm:
    def __init__(self, mu, sigma, population_size=50, generations=150, crossover_rate=0.8, mutation_rate=0.1, mutation_std=0.05, hc_probability=0.3, hc_iterations=10):
        # Class Constructor: initializes all hyperparameters of the algorithm
        # Settings are inspired by research papers to achieve good convergence

        self.mu = mu
        self.sigma = sigma
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.mutation_std = mutation_std
        self.hc_probability = hc_probability
        self.hc_iterations = hc_iterations
        self.n_assets = len(mu)

    def init_population(self):
        """
        Random generation of weights for each portfolio + normalization
        """
        population = []
        for _ in range(self.population_size):
            weights = np.random.rand(self.n_assets)
            weights = validate_and_normalize_weights(weights)
            population.append(weights)
        return population

    def tournament_selection(self, population, fitnesses, k=3):
        """
        Select a few individuals at random and return the best one
        """
        selected_indices = random.sample(range(len(population)), k)
        best_idx = max(selected_indices, key=lambda idx: fitnesses[idx])

        return population[best_idx]

    def arithmetic_crossover(self, parent1, parent2):
        """
        Combine 2 parents using a random alpha factor between 0 and 1
        """
        if random.random() < self.crossover_rate:
            alpha = random.random()
            offspring1 = alpha * parent1 + (1 - alpha) * parent2
            offspring2 = (1 - alpha) * parent1 + alpha * parent2

            return validate_and_normalize_weights(offspring1), validate_and_normalize_weights(offspring2)

        return parent1.copy(), parent2.copy()

    def gaussian_mutation(self, individual):
        """
        Add Gaussian noise to the genes of the individual
        """
        if random.random() < self.mutation_rate:
            noise = np.random.normal(0, self.mutation_std, self.n_assets)
            mutated = individual + noise
            return validate_and_normalize_weights(mutated)
        return individual

    def hill_climbing(self, individual):
        """
        Take a portfolio and adjust the weights to increase Sharpe Ratio
        """
        current_best = individual.copy()
        current_fitness = calculate_sharpe_ratio(current_best, self.mu, self.sigma)

        for _ in range(self.hc_iterations):
            neighbor = current_best.copy()

            # Randomly select 2 different assets
            idx1, idx2 = random.sample(range(self.n_assets), 2)

            # Move a small weight (e.g. between 1% and 5%) from asset 1 to asset 2
            shift_amount = random.uniform(0.01, 0.05)

            # Check that it doesn't go below zero
            if neighbor[idx1] >= shift_amount:
                neighbor[idx1] -= shift_amount
                neighbor[idx2] += shift_amount

                neighbor = validate_and_normalize_weights(neighbor)
                neighbor_fitness = calculate_sharpe_ratio(neighbor, self.mu, self.sigma)

                # If a better neighboring portfolio is found, keep it
                if neighbor_fitness > current_fitness:
                    current_best = neighbor.copy()
                    current_fitness = neighbor_fitness
        return current_best

    def run_memetic_algorithm(self, verbose=False):
        """
        Main loop for the Memetic Algorithm (HGA - Hybrid Genetic Algorithm)
        """
        population = self.init_population()

        best_fitness_history = []
        global_best_individual = None
        global_best_fitness = -np.inf

        if verbose:
            print(f"Starting memetic optimization for {self.generations} generations, Population: {self.population_size}:")

        for generation in range(self.generations):

            # Evaluate populations
            fitnesses = [calculate_sharpe_ratio(ind, self.mu, self.sigma) for ind in population]

            # Save the best individual from current generation
            gen_best_idx = np.argmax(fitnesses)
            if fitnesses[gen_best_idx] > global_best_fitness:
                global_best_fitness = fitnesses[gen_best_idx]
                global_best_individual = population[gen_best_idx].copy()

            best_fitness_history.append(global_best_fitness)

            new_population = []

            # The best individual is kept in the new population
            new_population.append(global_best_individual.copy())

            # Rest of the new population
            while len(new_population) < self.population_size:

                # Selection
                p1 = self.tournament_selection(population, fitnesses)
                p2 = self.tournament_selection(population, fitnesses)

                # Crossover
                offspring1, offspring2 = self.arithmetic_crossover(p1, p2)

                # Mutation
                offspring1 = self.gaussian_mutation(offspring1)
                offspring2 = self.gaussian_mutation(offspring2)

                # Local Search with HC
                if random.random() < self.hc_probability:
                    offspring1 = self.hill_climbing(offspring1)
                if random.random() < self.hc_probability:
                    offspring2 = self.hill_climbing(offspring2)

                new_population.append(offspring1)
                if len(new_population) < self.population_size:
                    new_population.append(offspring2)

            population = new_population

            if verbose and (generation + 1) % 25 == 0:
                print(f"Generation {generation + 1}/{self.generations} | Best Sharpe Ratio: {global_best_fitness:.4f}")

        return global_best_individual, global_best_fitness, best_fitness_history


In [4]:
class PSOAlgorithm:
    def __init__(self, mu, sigma, population_size=50, iterations=150, w=0.7298, c1=1.4962, c2=1.4962, early_stopping=True):
        """
        Class Constructor: Initializes the Particle Swarm Optimisation parameters.
        Parameters (w, c1, c2) are based on standard professional convergence coefficients
        (Clerc & Kennedy, 2002) to ensure swarm stability.
        """
        self.mu = mu
        self.sigma = sigma
        self.population_size = population_size
        self.iterations = iterations

        # PSO Specific Hyperparameters
        self.w = w    # Inertia weight: controls the impact of the previous velocity
        self.c1 = c1  # Cognitive coefficient: tendency to return to personal best
        self.c2 = c2  # Social coefficient: tendency to move toward global best

        self.early_stopping = early_stopping
        self.n_assets = len(mu)

    def init_particles(self):
        """
        Initialises particles with random weight distributions and zero initial velocities.
        """
        particles = []
        velocities = []
        for _ in range(self.population_size):
            # Initialise weights using Dirichlet to ensure they sum to 1 immediately
            weights = np.random.dirichlet(np.ones(self.n_assets))
            particles.append(weights)
            # Initialise velocity as small random values
            velocities.append(np.random.uniform(-0.01, 0.01, self.n_assets))

        return np.array(particles), np.array(velocities)

    def _should_stop(self, history):
        """
        Early stopping logic consistent with the SCA-BAS model.
        """
        if len(history) < 10:
            return False
        # Check if the improvement over the last 10 iterations is negligible
        return all(abs(history[-1] - v) < 1e-7 for v in history[-10:])

    def run_pso(self, verbose=False):
        """
        Main execution loop for the Particle Swarm Optimisation algorithm.
        Objective: Maximise the Sharpe Ratio through iterative swarm movement.
        """
        # Initialisation
        particles, velocities = self.init_particles()

        # Track personal bests for each particle
        p_best_positions = particles.copy()
        p_best_scores = np.array([calculate_sharpe_ratio(p, self.mu, self.sigma) for p in particles])

        # Track global best
        g_best_idx = np.argmax(p_best_scores)
        g_best_position = p_best_positions[g_best_idx].copy()
        g_best_score = p_best_scores[g_best_idx]

        history = [g_best_score]

        if verbose:
            print(f"Starting PSO Optimisation: {self.iterations} iterations, Swarm Size: {self.population_size}")

        for iteration in range(self.iterations):

            for i in range(self.population_size):
                # Random stochastic components
                r1 = np.random.rand(self.n_assets)
                r2 = np.random.rand(self.n_assets)

                # 1. Update Velocity
                # Formula: V(t+1) = w*V(t) + c1*r1*(pbest - X) + c2*r2*(gbest - X)
                cognitive = self.c1 * r1 * (p_best_positions[i] - particles[i])
                social = self.c2 * r2 * (g_best_position - particles[i])
                velocities[i] = (self.w * velocities[i]) + cognitive + social

                # 2. Update Position (Weights)
                particles[i] = particles[i] + velocities[i]

                # 3. Constraint Handling (Non-negativity and Sum-to-One)
                particles[i] = validate_and_normalize_weights(particles[i])

                # 4. Evaluate Performance
                current_score = calculate_sharpe_ratio(particles[i], self.mu, self.sigma)

                # 5. Update Personal Best
                if current_score > p_best_scores[i]:
                    p_best_scores[i] = current_score
                    p_best_positions[i] = particles[i].copy()

                    # 6. Update Global Best
                    if current_score > g_best_score:
                        g_best_score = current_score
                        g_best_position = particles[i].copy()

            # Record history for convergence analysis
            history.append(g_best_score)

            # Early Stopping Check
            if self.early_stopping and self._should_stop(history):
                if verbose:
                    print(f"Convergence achieved: Early stopping at iteration {iteration + 1}")
                break

            if verbose and (iteration + 1) % 25 == 0:
                print(f"Iteration {iteration + 1}/{self.iterations} | Global Best Sharpe: {g_best_score:.4f}")

        return g_best_position, g_best_score, history

In [5]:

def should_stop(history):
    if len(history) < 5:
        return False

    val = history[-1]
    vals = history[-5:]
    return all(abs(val - v) < 1e-6 for v in vals)

def sca_bas_portfolio_optimization(
        get_fitness,
        n_stocks=25,
        n_agents=25,
        max_iter=100,
        sensing_distance=0.1,
        step_size=0.1,
        early_stopping=True,
):
    X = np.random.dirichlet(np.ones(n_stocks), size=n_agents)

    p_best = X.copy()

    fitness_scores = np.array([get_fitness(ind) for ind in X])
    g_best = X[np.argmin(fitness_scores)].copy()

    history = [-get_fitness(g_best)]

    for t in range(max_iter):
        if early_stopping and should_stop(history):
            print(f"Early stopping at iteration {t}")
            break

        r1 = 2 * (1 - t / max_iter)
        omega = 1 / (1 + 1.5 * np.exp(10 * t / max_iter - 5)) - 0.1 * np.random.rand()

        for i in range(n_agents):
            r2 = 2 * np.pi * np.random.rand()
            r3 = 2 * np.pi * np.random.rand()
            r4 = np.random.rand()

            if r4 < 0.5:
                X[i] = omega * X[i] + r1 * np.sin(r2) * np.abs(r3 * g_best - X[i])
            else:
                X[i] = omega * X[i] + r1 * np.cos(r2) * np.abs(r3 * g_best - X[i])

            X[i] = np.clip(X[i], 0, 1)
            X[i] /= (np.sum(X[i]) + 1e-12)

            direction = np.random.randn(n_stocks)
            direction /= (np.linalg.norm(direction) + 1e-12)

            x_left = p_best[i] + sensing_distance * direction
            x_right = p_best[i] - sensing_distance * direction

            if get_fitness(x_left) < get_fitness(x_right):
                p_new = p_best[i] + step_size * direction
            else:
                p_new = p_best[i] - step_size * direction

            p_new = np.clip(p_new, 0, 1)
            p_new /= (np.sum(p_new) + 1e-12)

            if get_fitness(p_new) < get_fitness(p_best[i]):
                p_best[i] = p_new.copy()

            current_fit = get_fitness(X[i])
            if current_fit < get_fitness(p_best[i]):
                p_best[i] = X[i].copy()

            if get_fitness(p_best[i]) < get_fitness(g_best):
                g_best = p_best[i].copy()

        history.append(-get_fitness(g_best))

    return g_best, history

In [6]:
def compute_markowitz_benchmark(mu_train, sigma_train, mu_test, sigma_test):
    """
    Compute Markowitz benchmark on both training and test data.
    Returns weights found on training data along with both train and test Sharpe ratios and runtime.
    """
    start = time.perf_counter()
    optimal_weights, train_sharpe = calculate_markowitz_benchmark(mu_train, sigma_train)
    end = time.perf_counter()
    runtime = end - start

    test_sharpe = calculate_sharpe_ratio(optimal_weights, mu_test, sigma_test)

    return {
        'weights': optimal_weights,
        'train_sharpe': train_sharpe,
        'test_sharpe': test_sharpe,
        'runtime': runtime
    }

In [7]:

def run_hga_experiments():
    results_dir = os.path.join('results', 'results_hga')
    os.makedirs(results_dir, exist_ok=True)

    print("1. Loading data...")
    mu_train, sigma_train, mu_test, sigma_test = get_processed_data()

    print("2. Computing Benchmark (Markowitz)...")
    markowitz_result = compute_markowitz_benchmark(mu_train, sigma_train, mu_test, sigma_test)
    markowitz_train_sharpe = markowitz_result['train_sharpe']
    markowitz_test_sharpe = markowitz_result['test_sharpe']
    markowitz_runtime = markowitz_result['runtime']

    print("Markowitz (Mathematical Optimization) Results:")
    print(f"    Train Sharpe: {markowitz_train_sharpe:.4f}")
    print(f"    Test Sharpe: {markowitz_test_sharpe:.4f}")
    print(f"    Runtime: {markowitz_runtime:.4f}s")

    experiments = [
        {'name': 'HGA Standard', 'pop': 50, 'gen': 100, 'hc_prob': 0.3, 'hc_iter': 10},
        {'name': 'HGA Fast', 'pop': 30, 'gen': 50, 'hc_prob': 0.2, 'hc_iter': 5},
        {'name': 'HGA Aggressive', 'pop': 60, 'gen': 150, 'hc_prob': 0.5, 'hc_iter': 15}
    ]

    results_summary = [{
        'Experiment Name': 'Benchmark Markowitz',
        'Algorithm Settings': 'Mathematical Optimization',
        'Sharpe Ratio performance in Train': round(markowitz_train_sharpe, 4),
        'Sharpe Ratio performance in Test': round(markowitz_test_sharpe, 4),
        'Evolution of performance (Test vs Train)': f'{round(markowitz_test_sharpe - markowitz_train_sharpe, 4)}',
        'Runtime': f'{markowitz_runtime:.4f}s'
    }]

    experiments_names = ['Markowitz']
    train_scores = [markowitz_train_sharpe]
    test_scores = [markowitz_test_sharpe]
    convergence_histories = {}

    print("\n3. Running HGA Experiments")
    for exp in experiments:
        print(f"Running {exp['name']}...")
        algo = MemeticAlgorithm(
            mu=mu_train,
            sigma=sigma_train,
            population_size=exp['pop'],
            generations=exp['gen'],
            hc_probability=exp['hc_prob'],
            hc_iterations=exp['hc_iter']
        )
        start = time.perf_counter()
        best_weights, train_sharpe, history = algo.run_memetic_algorithm(verbose=False)
        end = time.perf_counter()
        runtime = end - start
        print(f"Time: {runtime:.4f}s")
        test_sharpe = calculate_sharpe_ratio(best_weights, mu_test, sigma_test)

        convergence_histories[exp['name']] = history
        experiments_names.append(exp['name'])
        train_scores.append(train_sharpe)
        test_scores.append(test_sharpe)

        results_summary.append({
            'Experiment Name': exp['name'],
            'Algorithm Settings': f"Population:{exp['pop']} Generations:{exp['gen']} HC_Probability:{exp['hc_prob']}",
            'Sharpe Ratio performance in Train': round(train_sharpe, 4),
            'Sharpe Ratio performance in Test': round(test_sharpe, 4),
            'Evolution of performance (Test vs Train)': f'{round(test_sharpe - train_sharpe, 4)}',
            'Runtime': f'{runtime:.4f}s'
        })
        print(f"Sharpe Ratio on train dataset: {train_sharpe:.4f}    Sharpe Ratio on test dataset: {test_sharpe:.4f}")

    df = pd.DataFrame(results_summary)
    csv_path = os.path.join(results_dir, 'report_hga.csv')
    df.to_csv(csv_path, index=False)
    print(f'Report saved to {csv_path}')

    print('Generating plots...')
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    # Plot 1: Convergence
    plt.figure(figsize=(9, 5))
    plt.axhline(y=markowitz_train_sharpe, color='black', linestyle='--', label='Markowitz (Mathematical Maximum)')

    for i, (name, history) in enumerate(convergence_histories.items()):
        plt.plot(history, label=name, color=colors[i], linewidth=2)

    plt.title('Memetic Algorithm Convergence Evolution (Training Data)', fontsize=12)
    plt.xlabel('Generations')
    plt.ylabel('Sharpe Ratio')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'convergence_plot_for_hga.png'))
    plt.close()

    # Plot 2: Train vs Test Comparison
    x = np.arange(len(experiments_names))
    width = 0.35

    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, train_scores, width, label='Train (In-Sample)', color='#7fbc41')
    plt.bar(x + width / 2, test_scores, width, label='Test (Out-of-Sample)', color='#de77ae')

    plt.ylabel('Sharpe Ratio')
    plt.title('In-Sample Performance vs Out-of-Sample Performance', fontsize=12)
    plt.xticks(x, experiments_names)
    plt.legend()
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'performance_plot_hga.png'))
    plt.close()

    print("All plots saved in results/results_hga/")


def run_pso_experiments():
    # Setup results directory
    results_dir = os.path.join('results', 'results_pso')
    os.makedirs(results_dir, exist_ok=True)

    print("1. Loading processed datasets...")
    mu_train, sigma_train, mu_test, sigma_test = get_processed_data()

    print("2. Computing Benchmark (Markowitz)...")
    markowitz_result = compute_markowitz_benchmark(mu_train, sigma_train, mu_test, sigma_test)
    markowitz_train_sharpe = markowitz_result['train_sharpe']
    markowitz_test_sharpe = markowitz_result['test_sharpe']
    markowitz_runtime = markowitz_result['runtime']

    print("Markowitz (Mathematical Optimization) Results:")
    print(f"    Train Sharpe: {markowitz_train_sharpe:.4f}")
    print(f"    Test Sharpe: {markowitz_test_sharpe:.4f}")
    print(f"    Runtime: {markowitz_runtime:.4f}s")

    # Define Experiment Configurations
    experiments = [
        {'name': 'PSO Fast', 'pop': 20, 'iter': 50, 'w': 0.7, 'c1': 1.2, 'c2': 1.2},
        {'name': 'PSO Standard', 'pop': 40, 'iter': 100, 'w': 0.7298, 'c1': 1.4962, 'c2': 1.4962},
        {'name': 'PSO Aggressive', 'pop': 70, 'iter': 200, 'w': 0.8, 'c1': 2.0, 'c2': 2.0}
    ]

    results_summary = [{
        'Experiment Name': 'Benchmark Markowitz',
        'Algorithm Settings': 'Mathematical Optimization',
        'Sharpe Ratio performance in Train': round(markowitz_train_sharpe, 4),
        'Sharpe Ratio performance in Test': round(markowitz_test_sharpe, 4),
        'Evolution of performance (Test vs Train)': f'{round(markowitz_test_sharpe - markowitz_train_sharpe, 4)}',
        'Runtime': f'{markowitz_runtime:.4f}s'
    }]

    experiments_names = ['Markowitz']
    train_scores = [markowitz_train_sharpe]
    test_scores = [markowitz_test_sharpe]
    convergence_histories = {}

    print("\n3. Running Particle Swarm Optimization Experiments")
    for exp in experiments:
        print(f"Running {exp['name']}...")

        # Initialize PSO Class
        algo = PSOAlgorithm(
            mu=mu_train,
            sigma=sigma_train,
            population_size=exp['pop'],
            iterations=exp['iter'],
            w=exp['w'],
            c1=exp['c1'],
            c2=exp['c2'],
            early_stopping=False
        )

        # Execute Algorithm
        start = time.perf_counter()
        best_weights, train_sharpe, history = algo.run_pso(verbose=False)
        end = time.perf_counter()
        runtime = end - start
        print(f"Time: {runtime:.4f}s")

        # Out-of-sample performance
        test_sharpe = calculate_sharpe_ratio(best_weights, mu_test, sigma_test)

        # Store results for report and plots
        convergence_histories[exp['name']] = history
        experiments_names.append(exp['name'])
        train_scores.append(train_sharpe)
        test_scores.append(test_sharpe)

        results_summary.append({
            'Experiment Name': exp['name'],
            'Algorithm Settings': f"Pop:{exp['pop']} Iter:{exp['iter']} w:{exp['w']} c1:{exp['c1']}",
            'Sharpe Ratio performance in Train': round(train_sharpe, 4),
            'Sharpe Ratio performance in Test': round(test_sharpe, 4),
            'Evolution of performance (Test vs Train)': f'{round(test_sharpe - train_sharpe, 4)}',
            'Runtime': f'{runtime:.4f}s'
        })

        print(f"   Done. Train Sharpe: {train_sharpe:.4f} | Test Sharpe: {test_sharpe:.4f}")

    # Export to CSV
    df = pd.DataFrame(results_summary)
    csv_path = os.path.join(results_dir, 'report_pso_experiments.csv')
    df.to_csv(csv_path, index=False)
    print(f'\nReport saved to: {csv_path}')

    # Visualization
    print('Generating plots...')
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    # Plot 1: Convergence Curves
    plt.figure(figsize=(10, 6))
    plt.axhline(y=markowitz_train_sharpe, color='black', linestyle='--', label='Markowitz (Mathematical Maximum)')

    for i, (name, history) in enumerate(convergence_histories.items()):
        plt.plot(history, label=name, color=colors[i % len(colors)], linewidth=2)

    plt.title('PSO Convergence Evolution (Training Data)', fontsize=14)
    plt.xlabel('Iterations')
    plt.ylabel('Sharpe Ratio')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'pso_convergence_evolution.png'))
    plt.close()

    # Plot 2: Train vs Test Comparison (Generalization)
    x = np.arange(len(experiments_names))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width / 2, train_scores, width, label='In-Sample (Train)', color='#7fbc41')
    plt.bar(x + width / 2, test_scores, width, label='Out-of-Sample (Test)', color='#de77ae')

    plt.ylabel('Sharpe Ratio')
    plt.title('PSO Generalization Performance: Train vs Test', fontsize=14)
    plt.xticks(x, experiments_names)
    plt.legend()
    plt.grid(axis='y', linestyle=':', alpha=0.4)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'pso_performance_comparison.png'))
    plt.close()

    print("All plots saved in results/results_pso/")


def run_sca_bas_experiments():
    results_dir = os.path.join('results', 'results_sca_bas')
    os.makedirs(results_dir, exist_ok=True)

    print("1. Loading data...")
    mu_train, sigma_train, mu_test, sigma_test = get_processed_data()

    print("2. Computing Benchmark (Markowitz)...")
    markowitz_result = compute_markowitz_benchmark(mu_train, sigma_train, mu_test, sigma_test)
    markowitz_train_sharpe = markowitz_result['train_sharpe']
    markowitz_test_sharpe = markowitz_result['test_sharpe']
    markowitz_runtime = markowitz_result['runtime']

    print("Markowitz (Mathematical Optimization) Results:")
    print(f"    Train Sharpe: {markowitz_train_sharpe:.4f}")
    print(f"    Test Sharpe: {markowitz_test_sharpe:.4f}")
    print(f"    Runtime: {markowitz_runtime:.4f}s")

    print("2. Computing Benchmark(Markowitz)...")
    markowitz_weights, markowitz_train_sharpe = calculate_markowitz_benchmark(mu_train, sigma_train)
    markowitz_test_sharpe = calculate_sharpe_ratio(markowitz_weights, mu_test, sigma_test)

    print(f"2. Markowitz(mathematical) results:")
    print(f"    Train:{markowitz_train_sharpe:.4f}")
    print(f"    Test:{markowitz_test_sharpe:.4f}")

    experiments = [
        {'name': 'SCA-BAS Fast', 'n_agents': 15, 'max_iter': 60, 'sensing_distance': 0.15, 'step_size': 0.15},
        {'name': 'SCA-BAS Standard', 'n_agents': 25, 'max_iter': 100, 'sensing_distance': 0.1, 'step_size': 0.1},
        {'name': 'SCA-BAS Aggressive', 'n_agents': 40, 'max_iter': 150, 'sensing_distance': 0.2, 'step_size': 0.2}
    ]

    results_summary = [{
        'Experiment Name': 'Benchmark Markowitz',
        'Algorithm Settings': 'Mathematical Optimization',
        'Sharpe Ratio performance in Train': round(markowitz_train_sharpe, 4),
        'Sharpe Ratio performance in Test': round(markowitz_test_sharpe, 4),
        'Evolution of performance (Test vs Train)': f'{round(markowitz_test_sharpe - markowitz_train_sharpe, 4)}',
        'Runtime': f'{markowitz_runtime:.4f}s'
    }]

    _,max_test = calculate_markowitz_benchmark(mu_test,sigma_test)

    results_summary.append({
        'Experiment Name': 'Benchmark Markowitz',
        'Algorithm Settings': 'Mathematical Optimization',
        'Sharpe Ratio performance in Test': max_test,
        'Evolution of performance (Test vs Train)': '-',
        'Runtime': '-',
    })


    experiments_names = ['Markowitz']
    train_scores = [markowitz_train_sharpe]
    test_scores = [markowitz_test_sharpe]
    convergence_histories = {}

    print("3. Running SCA-BAS Experiments")

    get_fitness_train = lambda weights: -calculate_sharpe_ratio(validate_and_normalize_weights(np.array(weights)), mu_train, sigma_train)

    for exp in experiments:
        print(f"Running {exp['name']}...")

        start = time.perf_counter()
        best_weights, history = sca_bas_portfolio_optimization(
            get_fitness=get_fitness_train,
            n_stocks=len(mu_train),
            n_agents=exp['n_agents'],
            max_iter=exp['max_iter'],
            sensing_distance=exp['sensing_distance'],
            step_size=exp['step_size'],
            early_stopping=False
        )
        end = time.perf_counter()
        runtime = end - start
        print(f"Time: {runtime:.4f}s")

        best_weights = validate_and_normalize_weights(best_weights)

        train_sharpe = calculate_sharpe_ratio(best_weights, mu_train, sigma_train)
        test_sharpe = calculate_sharpe_ratio(best_weights, mu_test, sigma_test)

        convergence_histories[exp['name']] = history
        experiments_names.append(exp['name'])
        train_scores.append(train_sharpe)
        test_scores.append(test_sharpe)

        results_summary.append({
            'Experiment Name': exp['name'],
            'Algorithm Settings': f"Agents:{exp['n_agents']}   Iterations:{exp['max_iter']}   Sensing:{exp['sensing_distance']}   Step:{exp['step_size']}",
            'Sharpe Ratio performance in Train': round(train_sharpe, 4),
            'Sharpe Ratio performance in Test': round(test_sharpe, 4),
            'Evolution of performance (Test vs Train)': f'{round(test_sharpe - train_sharpe, 4)}',
            'Runtime': f'{runtime:.4f}s'
        })

        print(f"Sharpe Ratio on train dataset: {train_sharpe:.4f}    Sharpe Ratio on test dataset: {test_sharpe:.4f}")

    df = pd.DataFrame(results_summary)
    csv_path = os.path.join(results_dir, 'report_sca_bas.csv')
    df.to_csv(csv_path, index=False)
    print(f'Report saved to {csv_path}')

    print('Generating plots...')
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    plt.figure(figsize=(9, 5))
    plt.axhline(y=markowitz_train_sharpe, color='black', linestyle='--', label='Markowitz (Mathematical Maximum)')

    for i, (name, history) in enumerate(convergence_histories.items()):
        plt.plot(history, label=name, color=colors[i % len(colors)], linewidth=2)

    plt.title('SCA-BAS Convergence Evolution (Training Data)', fontsize=12)
    plt.xlabel('Iterations')
    plt.ylabel('Sharpe Ratio')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'convergence_plot_for_sca_bas.png'))
    plt.close()

    x = np.arange(len(experiments_names))
    width = 0.35

    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, train_scores, width, label='Train (In-Sample)', color='#7fbc41')
    plt.bar(x + width / 2, test_scores, width, label='Test (Out-of-Sample)', color='#de77ae')

    plt.ylabel('Sharpe Ratio')
    plt.title('In-Sample Performance vs Out-of-Sample Performance', fontsize=12)
    plt.xticks(x, experiments_names)
    plt.legend()
    plt.grid(axis='y', linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'performance_plot_sca_bas.png'))
    plt.close()

    print("All plots saved in results/results_sca_bas/")

In [8]:
run_hga_experiments()

1. Loading data...
Loading processed data from local 'data/' folder...
2. Computing Benchmark (Markowitz)...
Markowitz (Mathematical Optimization) Results:
    Train Sharpe: 1.8847
    Test Sharpe: 1.4583
    Runtime: 0.3875s

3. Running HGA Experiments
Running HGA Standard...
Time: 0.1347s
Sharpe Ratio on train dataset: 1.8248    Sharpe Ratio on test dataset: 1.4576
Running HGA Fast...
Time: 0.0235s
Sharpe Ratio on train dataset: 1.7101    Sharpe Ratio on test dataset: 1.4161
Running HGA Aggressive...
Time: 0.3089s
Sharpe Ratio on train dataset: 1.8654    Sharpe Ratio on test dataset: 1.4522
Report saved to results/results_hga/report_hga.csv
Generating plots...
All plots saved in results/results_hga/
